In [ ]:
import numpy as np
import utils
from MfccFixedRandomForest import MfccFixedRandomForest

In [ ]:
# Load metadata and features.

tracks = utils.load('data/fma_metadata/tracks.csv')
genres = utils.load('data/fma_metadata/genres.csv')
features = utils.load('data/fma_metadata/features.csv')
echonest = utils.load('data/fma_metadata/echonest.csv')

In [ ]:
def get_top_level_name(genre_id):
    row = genres.loc[genre_id]
    if row['parent'] == 0:  # 0 means top-level
        return row['title']
    return get_top_level_name(row['parent'])

genre_to_root_map = {gid: get_top_level_name(gid) for gid in genres.index}

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np

# 1. Use the 'medium' subset for a better balance of data and cleanup
is_medium = tracks['set', 'subset'] <= 'medium'

# 2. Extract the most specific genre (subgenre) for each track
# This uses the full taxonomy instead of the top-level 'genre_top'
y_full = tracks.loc[is_medium, ('track', 'genres')].apply(
    lambda x: get_top_level_name(x[0]) if len(x) > 0 else None
)

# 3. Create masks specific to the medium subset
train_mask = (tracks['set', 'split'] == 'training') & is_medium
test_mask = (tracks['set', 'split'] == 'test') & is_medium

# 4. Extract features and labels
X_train = features.loc[train_mask].values
X_test = features.loc[test_mask].values

y_train = y_full.loc[train_mask]
y_test = y_full.loc[test_mask]

# 5. Data Cleaning: Remove tracks with NaNs or missing labels
train_valid = ~np.isnan(X_train).any(axis=1) & y_train.notna()
test_valid = ~np.isnan(X_test).any(axis=1) & y_test.notna()

X_train, y_train = X_train[train_valid], y_train[train_valid]
X_test, y_test = X_test[test_valid], y_test[test_valid]

# 6. Standardize features
# Note: StandardScaler is fit ONLY on training data to prevent leakage
scaler = StandardScaler(copy=False)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 7. Encode the subgenre IDs into integers
le = LabelEncoder()
le.fit(y_full.dropna())
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

print(f"Training on {len(y_train_enc)} samples across {len(le.classes_)} unique subgenres.")

In [ ]:
 # Initialize Categorical RF
# Setting features_per_cat=4 gives each tree 20 total features (4 from 5 groups)
model = MfccFixedRandomForest(n_estimators=1500, features_per_other_cat=30)

print("Starting training on the medium dataset...")
model.fit(X_train, y_train_enc)

print("Calculating accuracy...")
y_pred = model.predict(X_test)
accuracy = np.mean(y_pred == y_test_enc)

print(f"Initial Accuracy on medium Set: {accuracy:.2%}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Initialize the standard Scikit-Learn Random Forest
standard_rf = RandomForestClassifier(
    n_estimators=2000,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=5
)

# 2. Fit on the same Medium dataset (Top-Level)
print("Training standard RandomForest baseline...")
standard_rf.fit(X_train, y_train_enc)

# 3. Predict and Compare
y_pred_std = standard_rf.predict(X_test)
std_accuracy = accuracy_score(y_test_enc, y_pred_std)

print(f"Standard RF Accuracy: {std_accuracy:.2%}")

# 4. View detailed comparison
print("\nStandard RF Classification Report:\n")
print(classification_report(y_test_enc, y_pred_std, target_names=le.classes_))

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

ann_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=32,
    learning_rate='adaptive',
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    verbose=True,
    tol=0.00001
)

# Train the model
print("Training MLP (ANN)...")
ann_model.fit(X_train, y_train_enc)

# Predict and Evaluate
y_pred_ann = ann_model.predict(X_test)
ann_accuracy = accuracy_score(y_test_enc, y_pred_ann)

print(f"ANN Test Accuracy: {ann_accuracy:.2%}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred = ann_model.predict(X_test)

cm = confusion_matrix(y_test_enc, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.title('Genre Classification Confusion Matrix')
plt.xlabel('Predicted Genre')
plt.ylabel('True Genre')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig('images/ann/cm_ann_small_tol.png')

print("metrics:")
print(classification_report(y_test_enc, y_pred))


In [10]:
def get_specialist_data(root_genre_name, tracks_df, features_df):
    subset_mask = (tracks_df['set', 'subset'] <= 'medium')

    roots = tracks_df.loc[subset_mask, ('track', 'genres')].apply(
        lambda x: genre_to_root_map[x[0]] if len(x) > 0 else None
    )

    specialist_mask = (roots == root_genre_name)

    y_specialist = tracks_df.loc[subset_mask].loc[specialist_mask, ('track', 'genres')].apply(lambda x: x[0])
    X_specialist = features_df.loc[y_specialist.index].values

    return X_specialist, y_specialist

X_rock, y_rock = get_specialist_data('Rock', tracks, features)

In [12]:
# 1. Initialize y_all: Map every track's first genre ID to its Root Genre
y_all = tracks['track', 'genres'].apply(
    lambda x: genre_to_root_map[x[0]] if len(x) > 0 else None
)

# 2. Filter for only the Medium subset to match your features
is_medium = tracks['set', 'subset'] <= 'medium'
y_all_medium = y_all.loc[is_medium]

In [15]:
# 1. Filter for the Rock Root Genre
is_rock = (tracks['set', 'subset'] <= 'medium') & (y_all == 'Rock')

# 2. Get features and sub-genre labels
X_rock = features.loc[is_rock].values
y_rock_sub = tracks.loc[is_rock, ('track', 'genres')].apply(lambda x: x[0])

# 3. Create a fresh split for the specialist
train_mask_rock = (tracks.loc[is_rock, ('set', 'split')] == 'training')
test_mask_rock = (tracks.loc[is_rock, ('set', 'split')] == 'test')

X_train_rock = X_rock[train_mask_rock.values]
X_test_rock = X_rock[test_mask_rock.values]
y_train_rock = y_rock_sub[train_mask_rock.values]
y_test_rock = y_rock_sub[test_mask_rock.values]

# 4. Local Scaling (Crucial for specialists)
scaler_rock = StandardScaler()
X_train_rock = scaler_rock.fit_transform(X_train_rock)
X_test_rock = scaler_rock.transform(X_test_rock)

le_rock = LabelEncoder()
le_rock.fit(y_rock_sub.dropna())
y_train_rock_enc = le_rock.transform(y_train_rock)
y_test_rock_enc = le_rock.transform(y_test_rock)

print(f"Rock Specialist trained on {len(le_rock.classes_)} sub-genres.")

Rock Specialist trained on 29 sub-genres.


In [19]:
from sklearn.ensemble import RandomForestClassifier

# Use a high number of estimators to catch subtle sub-genre differences
rock_specialist = RandomForestClassifier(
    n_estimators=1500,
    class_weight='balanced',
    max_features='sqrt',
    n_jobs=-1
)

rock_specialist.fit(X_train_rock, y_train_rock_enc)
y_pred_rock = rock_specialist.predict(X_test_rock)

rock_acc = accuracy_score(y_test_rock_enc, y_pred_rock)
print(f"Rock Specialist Internal Accuracy: {rock_acc:.2%}")

Rock Specialist Internal Accuracy: 36.85%


In [20]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

top_10_roots = [
    'Rock', 'Electronic', 'Experimental', 'Hip-Hop', 'Folk',
    'Instrumental', 'Pop', 'Jazz', 'Classical', 'International'
]

# Dictionaries to store our models and encoders
specialist_models = {}
specialist_label_encoders = {}
specialist_scalers = {}

for root in top_10_roots:
    print(f"--- Training Specialist for: {root} ---")

    # 1. Filter tracks where this root is the top-level genre
    is_medium = tracks['set', 'subset'] <= 'medium'
    # Identify root for each track to find the members of this 'family'
    current_root_mask = tracks.loc[is_medium, ('track', 'genres')].apply(
        lambda x: genre_to_root_map[x[0]] if len(x) > 0 else None
    ) == root

    # 2. Extract specific sub-genre labels (the first ID in the list)
    y_subs = tracks.loc[is_medium].loc[current_root_mask, ('track', 'genres')].apply(lambda x: x[0])
    X_subs = features.loc[y_subs.index].values

    # 3. Handle splits
    split = tracks.loc[y_subs.index, ('set', 'split')]
    train_idx = split == 'training'
    test_idx = split == 'test'

    # Skip if not enough data for a split
    if train_idx.sum() < 10 or test_idx.sum() < 2:
        print(f"Skipping {root}: Not enough data.")
        continue

    X_train_sub, X_test_sub = X_subs[train_idx], X_subs[test_idx]
    y_train_sub, y_test_sub = y_subs[train_idx], y_subs[test_idx]

    # 4. Local Scaling
    scaler = StandardScaler()
    X_train_sub = scaler.fit_transform(X_train_sub)
    X_test_sub = scaler.transform(X_test_sub)

    # 5. Label Encoding
    le = LabelEncoder()
    # Using fit on all seen sub-genres for this root to avoid 'unseen label' errors
    y_train_enc = le.fit_transform(y_train_sub)

    # Filter test set for labels that actually exist in training
    valid_test_mask = y_test_sub.isin(le.classes_)
    X_test_sub = X_test_sub[valid_test_mask]
    y_test_sub = y_test_sub[valid_test_mask]
    y_test_enc = le.transform(y_test_sub)

    # 6. Train the Specialist
    rf = RandomForestClassifier(n_estimators=1000, class_weight='balanced', n_jobs=-1)
    rf.fit(X_train_sub, y_train_enc)

    # Store everything
    specialist_models[root] = rf
    specialist_label_encoders[root] = le
    specialist_scalers[root] = scaler

    acc = rf.score(X_test_sub, y_test_enc)
    print(f"Done. {root} Specialist Accuracy: {acc:.2%}")

--- Training Specialist for: Rock ---
Done. Rock Specialist Accuracy: 36.07%
--- Training Specialist for: Electronic ---
Done. Electronic Specialist Accuracy: 68.51%
--- Training Specialist for: Experimental ---
Done. Experimental Specialist Accuracy: 33.78%
--- Training Specialist for: Hip-Hop ---
Done. Hip-Hop Specialist Accuracy: 88.64%
--- Training Specialist for: Folk ---
Done. Folk Specialist Accuracy: 88.82%
--- Training Specialist for: Instrumental ---
Done. Instrumental Specialist Accuracy: 21.26%
--- Training Specialist for: Pop ---
Done. Pop Specialist Accuracy: 68.91%
--- Training Specialist for: Jazz ---
Done. Jazz Specialist Accuracy: 69.23%
--- Training Specialist for: Classical ---
Done. Classical Specialist Accuracy: 19.35%
--- Training Specialist for: International ---
Done. International Specialist Accuracy: 33.33%


In [ ]:
def predict_full_hierarchy(X_input):
    # 1. Router Prediction (Root Genre)
    # Ensure X_input is scaled using your GLOBAL scaler first
    X_scaled_global = global_scaler.transform(X_input)
    root_idx = router_model.predict(X_scaled_global)[0]
    root_name = le_global.inverse_transform([root_idx])[0]

    # 2. Specialist Prediction (Subgenre)
    if root_name in specialist_models:
        # Use the specialist's LOCAL scaler
        X_scaled_local = specialist_scalers[root_name].transform(X_input)

        # Get subgenre prediction
        sub_idx = specialist_models[root_name].predict(X_scaled_local)[0]
        sub_name = specialist_label_encoders[root_name].inverse_transform([sub_idx])[0]

        return root_name, sub_name
    else:
        # If no specialist exists (one of the bottom 6), return the root only
        return root_name, "No subgenre specialist available"

# Test it out!
root, sub = predict_full_hierarchy(X_test[0:1])
print(f"Final Prediction: {root} -> {sub}")